# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Sarah Vafiadis, Nathan Rhoads

**ID**: sv439

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\sarah\OneDrive\Documents\GitHub\hw5-sarah_nathan_hw5`
   Installed Measures ─────────── v0.3.3
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed GR_jll ───────────── v0.73.18+0
   Installed PlotUtils ────────── v1.4.4
   Installed OpenSSL ──────────── v1.6.0
   Installed MutableArithmetics ─ v1.6.7
   Installed FFMPEG ───────────── v0.4.5
   Installed Pango_jll ────────── v1.57.0+0
   Installed METIS_jll ────────── v5.1.3+0
   Installed StaticArraysCore ─── v1.4.4
   Installed JSON ─────────────── v1.3.0
   Installed DataStructures ───── v0.19.3
   Installed StatsBase ────────── v0.34.8
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed HiGHS ────────────── v1.20.1
   Installed StableRNGs ───────── v1.0.4
   Installed StructUtils ──────── v2.6.0
   Installed ForwardDiff ──────── v1.3.0
   Installed GR ───────────────── v0.73.18
   Installed JuMP ─────────────── v1.29.3
Precompiling project...
   8

In [1]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [ ]:
# Data

# City waste generation (mg/d)
cities = 1:3
waste = Dict(1 => 100.0, 2 => 90.0, 3 => 120.0)

# Distances from facilities to cities (km)
dist_city = Dict(
    :LF  => Dict(1 => 5.0,  2 => 15.0, 3 => 13.0),
    :MRF => Dict(1 => 30.0, 2 => 25.0, 3 => 45.0),
    :WTE => Dict(1 => 15.0, 2 => 10.0, 3 => 20.0)
)

# Distances between facilities (km)
dist_fac = Dict(
    (:MRF, :LF)  => 32.0,
    (:MRF, :WTE) => 15.0,
    (:WTE, :LF)  => 18.0
)

# Capacities (Mg/day)
cap = Dict(:LF => 200.0, :MRF => 350.0, :WTE => 210.0)

# Fixed costs ($/day)
F = Dict(:LF => 2000.0, :MRF => 1500.0, :WTE => 2500.0)

# Tipping costs ($/mg)
tipping = Dict(:LF => 50.0, :MRF => 7.0, :WTE => 60.0)

transport = 1.5     # Transportation costs ($/mg-km)
recycle = 40.0      # Recycling costs ($/mg recycled)

MRF_recycle = 0.4   # MRF recycling rate
ash_NR = 0.16       # Ash fraction of non-recycled waste
ash_recycle = 0.14  # Ash fraction of recycled waste

# Composition table (mass share, ash %, MRF recycle %)
# Component => (mass_share, ash_frac, recycle_frac)
comp = Dict(
    :Food            => (0.15, 0.08, 0.00),
    :PaperCardboard  => (0.40, 0.07, 0.55),
    :Plastics        => (0.05, 0.05, 0.15),
    :Textiles        => (0.03, 0.10, 0.10),
    :RubberLeather   => (0.02, 0.15, 0.00),
    :Wood            => (0.05, 0.02, 0.30),
    :Yard            => (0.18, 0.02, 0.40),
    :Glass           => (0.04, 1.00, 0.60),
    :Ferrous         => (0.02, 1.00, 0.75),
    :Aluminum        => (0.02, 1.00, 0.80),
    :OtherMetal      => (0.01, 1.00, 0.50),
    :Misc            => (0.03, 0.70, 0.00)
)







Dict{Symbol, Tuple{Float64, Float64, Float64}} with 12 entries:
  :Plastics       => (0.05, 0.05, 0.15)
  :Aluminum       => (0.02, 1.0, 0.8)
  :Glass          => (0.04, 1.0, 0.6)
  :Textiles       => (0.03, 0.1, 0.1)
  :RubberLeather  => (0.02, 0.15, 0.0)
  :PaperCardboard => (0.4, 0.07, 0.55)
  :Yard           => (0.18, 0.02, 0.4)
  :Ferrous        => (0.02, 1.0, 0.75)
  :OtherMetal     => (0.01, 1.0, 0.5)
  :Misc           => (0.03, 0.7, 0.0)
  :Wood           => (0.05, 0.02, 0.3)
  :Food           => (0.15, 0.08, 0.0)

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.